In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    TimestampType
)

# ============================================================
# 1. CONFIGURATION
# ============================================================

storage_account = "stsupplychain2026xx"

landing_events_path = (
    f"abfss://logistics@{storage_account}.dfs.core.windows.net/"
    "landing/delivery_events/"
)

bronze_events_path = (
    f"abfss://logistics@{storage_account}.dfs.core.windows.net/"
    "bronze/delivery_events/"
)

checkpoint_events_path = (
    f"abfss://logistics@{storage_account}.dfs.core.windows.net/"
    "checkpoints/bronze_delivery_events/"
)

schema_events_path = (
    f"abfss://logistics@{storage_account}.dfs.core.windows.net/"
    "checkpoints/schemas/bronze_delivery_events/"
)

# ============================================================
# 2. SCHÉMA SOURCE
# ============================================================

delivery_events_schema = StructType([
    StructField("event_id", StringType(), True),
    StructField("shipment_id", StringType(), True),
    StructField("event_time", TimestampType(), True),
    StructField("event_status", StringType(), True),
    StructField("location_city", StringType(), True),
    StructField("location_country", StringType(), True),
    StructField("source_system", StringType(), True),
    StructField("updated_at", TimestampType(), True)
])

# ============================================================
# 3. LECTURE INCRÉMENTALE AVEC AUTO LOADER
# ============================================================

events_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", schema_events_path)
    .schema(delivery_events_schema)
    .load(landing_events_path)
)

# ============================================================
# 4. AJOUT DES MÉTADONNÉES BRONZE
# ============================================================

bronze_events_df = (
    events_stream
    .withColumn(
        "source_file",
        F.col("_metadata.file_path")
    )
    .withColumn(
        "ingestion_timestamp",
        F.current_timestamp()
    )
    .withColumn(
        "batch_id",
        F.regexp_extract(
            F.col("_metadata.file_path"),
            r"(batch_\d+)",
            1
        )
    )
)

# ============================================================
# 5. ÉCRITURE DELTA
# ============================================================

query = (
    bronze_events_df
    .writeStream
    .format("delta")
    .outputMode("append")
    .option(
        "checkpointLocation",
        checkpoint_events_path
    )
    .option(
        "path",
        bronze_events_path
    )
    .trigger(availableNow=True)
    .start()
)

query.awaitTermination()

print("Ingestion Bronze delivery_events terminée.")

# ============================================================
# 6. LECTURE DE LA BRONZE
# ============================================================

bronze_events = (
    spark.read
    .format("delta")
    .load(bronze_events_path)
)

print(
    "Nombre de lignes Bronze delivery_events :",
    bronze_events.count()
)

display(
    bronze_events
    .orderBy(
        "shipment_id",
        "event_time"
    )
)

# ============================================================
# 7. VÉRIFICATION DES MÉTADONNÉES
# ============================================================

display(
    bronze_events.select(
        "event_id",
        "shipment_id",
        "event_status",
        "source_file",
        "batch_id",
        "ingestion_timestamp"
    )
    .orderBy(
        "shipment_id",
        "event_time"
    )
)